# CSE457 – Lab 2: Đặc trưng tiếng nói và nhận dạng bằng DTW

- **Họ tên:** ........................  **MSSV:** ........................
- **Bài toán:** nhận dạng 5 từ tiếng Việt tách rời (*không, một, hai, ba, bốn*) bằng MFCC + DTW tự cài đặt.
- **Mapping nhãn ASCII → tiếng Việt:** `khong` = không, `mot` = một, `hai` = hai, `ba` = ba, `bon` = bốn.

Pipeline: `WAV → endpoint detection → frame + Hamming → MFCC → DTW với templates → nhãn dự đoán`.

In [ ]:
import warnings
from pathlib import Path
from functools import lru_cache

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import soundfile as sf
from scipy.signal import lfilter
from sklearn.metrics import confusion_matrix, accuracy_score

warnings.filterwarnings("ignore")

# ---------------- Tham số (giữ giống nhau cho train/test) ----------------
FS = 16000
FRAME_MS, HOP_MS = 25, 10
WIN = int(round(FS * FRAME_MS / 1000))   # 400 mẫu
HOP = int(round(FS * HOP_MS / 1000))     # 160 mẫu
PREEMPH = 0.97
NFFT = 512
N_MELS = 24
N_MFCC = 13

DATASET = Path("dataset")
FIG = Path("figures");   FIG.mkdir(exist_ok=True)
TRIMMED = Path("trimmed"); TRIMMED.mkdir(exist_ok=True)

LABELS = ["khong", "mot", "hai", "ba", "bon"]
VN = {"khong": "không", "mot": "một", "hai": "hai", "ba": "ba", "bon": "bốn"}
N_TRAIN = 3   # 3 file đầu = template, các file còn lại = test

# Tham số endpoint detection (PHẢI báo cáo trong phần C)
EP_REL_DB = 30        # ngưỡng = đỉnh log-energy - 30 dB (và >= nền + 10 dB)
EP_MARGIN_FRAMES = 5  # đệm 5 frame = 50 ms mỗi phía

print(f"L = {WIN} mẫu, R = {HOP} mẫu, chồng lấn {WIN-HOP} mẫu = {(WIN-HOP)/FS*1000:.0f} ms")

## A. Thu dữ liệu và kiểm tra chất lượng

- Định dạng WAV mono 16 kHz, mỗi từ ≥ 5 lần lặp.
- **Chia train/test trước khi chọn template**: 3 file đầu của mỗi từ → template (train), các file còn lại → test. Không file nào dùng cho cả hai.

In [ ]:
def load_audio(path, normalize=True):
    # librosa tự resample về FS và đưa về mono
    y, _ = librosa.load(str(path), sr=FS, mono=True)
    if normalize:
        y = y / (np.max(np.abs(y)) + 1e-9)
    return y

def list_files(lab, root=DATASET):
    return sorted((Path(root) / lab).glob("*.wav"))

train_files = {lab: list_files(lab)[:N_TRAIN] for lab in LABELS}
test_files  = {lab: list_files(lab)[N_TRAIN:] for lab in LABELS}

# kiểm tra không rò rỉ dữ liệu
all_train = {f for v in train_files.values() for f in v}
all_test  = {f for v in test_files.values() for f in v}
assert all_train.isdisjoint(all_test), "Data leakage: file vừa train vừa test!"

split = pd.DataFrame({
    "label": LABELS,
    "n_train": [len(train_files[l]) for l in LABELS],
    "n_test":  [len(test_files[l]) for l in LABELS],
})
split

In [ ]:
# Kiểm tra chất lượng: độ dài, đỉnh, clipping, silence đầu/cuối
rows = []
for lab in LABELS:
    for f in list_files(lab):
        raw, _ = librosa.load(str(f), sr=FS, mono=True)
        _, (s, e) = librosa.effects.trim(raw, top_db=35, frame_length=WIN, hop_length=HOP)
        rows.append(dict(file=f.name, dur_s=len(raw)/FS, peak=np.max(np.abs(raw)),
                         clip_pct=100*np.mean(np.abs(raw) >= 0.99),
                         lead_sil_s=s/FS, trail_sil_s=(len(raw)-e)/FS))
quality = pd.DataFrame(rows)
quality.round(3).head(10)

In [ ]:
# Waveform của 3 từ khác nhau
fig, axes = plt.subplots(3, 1, figsize=(10, 7))
for ax, lab in zip(axes, LABELS[:3]):
    y = load_audio(train_files[lab][0])
    ax.plot(np.arange(len(y))/FS, y, lw=0.6)
    ax.set_title(f"{train_files[lab][0].name}  ({VN[lab]})")
    ax.set_ylabel("Biên độ"); ax.grid(alpha=.3)
axes[-1].set_xlabel("Thời gian (s)")
plt.tight_layout(); plt.savefig(FIG/"A_waveforms.png", dpi=150); plt.show()

print("Tổng hợp chất lượng toàn bộ dữ liệu:")
print(quality[["dur_s", "clip_pct", "lead_sil_s", "trail_sil_s"]].describe().round(3))

**Nhận xét A:** (điền theo kết quả của bạn) — Có file nào clipping (`clip_pct` > 0) không? Silence đầu/cuối trung bình là bao nhiêu giây, có nằm trong khoảng 0.2–0.5 s như quy ước không?

## B. Đặc trưng miền thời gian

Frame 25 ms, hop 10 ms, cửa sổ Hamming.

- $E_r=\sum x_r^2[n]$, $\mathrm{RMS}_r=\sqrt{E_r/L}$, $E_r(dB)=10\log_{10}(E_r+\varepsilon)$
- $Z_r=\frac{1}{2L}\sum|\mathrm{sgn}(x[m])-\mathrm{sgn}(x[m-1])|$ (cửa sổ chữ nhật)

In [ ]:
def frame_signal(y, win=WIN, hop=HOP):
    if len(y) < win:
        y = np.pad(y, (0, win - len(y)))
    n = 1 + (len(y) - win) // hop
    idx = np.arange(win)[None, :] + hop * np.arange(n)[:, None]
    return y[idx]                      # (n_frames, win)

def short_time_energy(y):
    fr = frame_signal(y) * np.hamming(WIN)      # Hamming: 0.54-0.46cos(2*pi*n/(L-1))
    return np.sum(fr ** 2, axis=1)

def short_time_rms(y):
    return np.sqrt(short_time_energy(y) / WIN)

def log_energy_db(E, eps=1e-10):
    return 10 * np.log10(E + eps)

def zero_crossing_rate(y):
    fr = frame_signal(y)                         # cửa sổ chữ nhật
    s = np.where(fr >= 0, 1, -1)
    return np.sum(np.abs(np.diff(s, axis=1)), axis=1) / (2 * WIN)

def frame_times(n_frames):
    return (np.arange(n_frames) * HOP + WIN / 2) / FS

# kiểm tra nhanh theo ví dụ trong đề: 100 mẫu, 20 lần đổi dấu → ZCR = 0.2
_t = np.tile([1.0, 1.0, 1.0, 1.0, -1.0, -1.0, -1.0, -1.0, -1.0, 1.0], 10)  # 100 mẫu
print("ZCR kiểm tra:", np.sum(np.abs(np.diff(np.where(_t >= 0, 1, -1)))) / 2 / len(_t))

In [ ]:
def plot_time_features(y, title, fname=None):
    E = short_time_energy(y); Edb = log_energy_db(E); Z = zero_crossing_rate(y)
    t = frame_times(len(E))
    fig, ax = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
    ax[0].plot(np.arange(len(y))/FS, y, lw=.6); ax[0].set_ylabel("Biên độ"); ax[0].set_title(title)
    ax[1].plot(t, Edb); ax[1].set_ylabel("Log-energy (dB)")
    ax[2].plot(t, Z);   ax[2].set_ylabel("ZCR"); ax[2].set_xlabel("Thời gian (s)")
    for a in ax: a.grid(alpha=.3)
    plt.tight_layout()
    if fname: plt.savefig(FIG/fname, dpi=150)
    plt.show()

for k, lab in enumerate(LABELS[:3]):
    f = train_files[lab][0]
    plot_time_features(load_audio(f), f"{f.name} ({VN[lab]})", f"B_energy_zcr_{lab}.png")

**Nhận xét B (điền theo đồ thị của bạn):**
- *Silence:* log-energy thấp và phẳng (chỉ là nhiễu nền), ZCR nhỏ/ngẫu nhiên tuỳ nhiễu.
- *Voiced (nguyên âm, âm hữu thanh):* energy cao, tuần hoàn, ZCR thấp.
- *Unvoiced/fricative (ví dụ "kh" trong *không*):* energy thấp hơn voiced nhưng cao hơn nền, ZCR cao do tín hiệu giống nhiễu băng rộng.

## C. Endpoint detection

Thuật toán (tự cài):
1. Tính log-energy và ZCR theo frame.
2. Ước lượng nền: trung vị 10% frame có năng lượng thấp nhất.
3. Ngưỡng energy: `thr = max(nền + 10 dB, đỉnh − EP_REL_DB)`; start/end thô = frame đầu/cuối vượt ngưỡng.
4. Tinh chỉnh bằng ZCR: lùi start / tiến end tối đa 15 frame khi ZCR còn cao hơn ngưỡng ZCR của nền (bắt phụ âm vô thanh yếu).
5. Thêm `EP_MARGIN_FRAMES` frame đệm hai phía.

In [ ]:
def endpoint_detect(y, rel_db=EP_REL_DB, margin_frames=EP_MARGIN_FRAMES,
                    zcr_refine=True, max_ext=15):
    E = log_energy_db(short_time_energy(y)); Z = zero_crossing_rate(y)
    n = len(E)
    srt = np.sort(E)
    noise = np.median(srt[:max(3, n // 10)])
    peak = E.max()
    thr = max(noise + 10.0, peak - rel_db)
    above = np.where(E > thr)[0]
    info = dict(noise_db=noise, peak_db=peak, thr_db=thr, E=E, Z=Z)
    if len(above) == 0:                       # không phát hiện tiếng nói
        info.update(s_frame=0, e_frame=n - 1, zthr=np.nan)
        return 0, len(y), info
    s0, e0 = above[0], above[-1]
    s, e = s0, e0
    # ngưỡng ZCR từ các frame có năng lượng gần nền
    sil = Z[E <= noise + 3.0]
    zthr = max(0.10, sil.mean() + 3 * sil.std()) if len(sil) > 3 else 0.2
    if zcr_refine:
        k = 0
        while s > 0 and k < max_ext and Z[s - 1] > zthr:
            s -= 1; k += 1
        k = 0
        while e < n - 1 and k < max_ext and Z[e + 1] > zthr:
            e += 1; k += 1
    s_m = max(0, s - margin_frames)
    e_m = min(n - 1, e + margin_frames)
    start = s_m * HOP
    end = min(len(y), e_m * HOP + WIN)
    info.update(s_frame=s_m, e_frame=e_m, s_raw=s0, e_raw=e0, zthr=zthr)
    return start, end, info

def trim_endpoint(y, **kw):
    s, e, _ = endpoint_detect(y, **kw)
    return y[s:e]

print(f"Tham số đã dùng: EP_REL_DB={EP_REL_DB} dB, margin={EP_MARGIN_FRAMES} frame ({EP_MARGIN_FRAMES*HOP_MS} ms), "
      f"nền+10 dB, ZCR refine tối đa 15 frame")

In [ ]:
# Trực quan hoá endpoint cho 3 file
fig, axes = plt.subplots(3, 1, figsize=(10, 8))
for ax, lab in zip(axes, LABELS[:3]):
    f = train_files[lab][0]; y = load_audio(f)
    s, e, info = endpoint_detect(y)
    ax.plot(np.arange(len(y))/FS, y, lw=.6)
    ax.axvline(s/FS, color="g"); ax.axvline(e/FS, color="r")
    ax.set_title(f"{f.name}: start={s/FS:.2f}s, end={e/FS:.2f}s, ngưỡng={info['thr_db']:.1f} dB")
    ax.grid(alpha=.3)
axes[-1].set_xlabel("Thời gian (s)")
plt.tight_layout(); plt.savefig(FIG/"C_endpoint.png", dpi=150); plt.show()

# Xuất file trim và so sánh thời lượng trước/sau
rows = []
for lab in LABELS:
    for f in list_files(lab):
        y = load_audio(f); yt = trim_endpoint(y)
        (TRIMMED/lab).mkdir(exist_ok=True, parents=True)
        sf.write(TRIMMED/lab/f.name, yt, FS)
        rows.append(dict(file=f.name, before_s=len(y)/FS, after_s=len(yt)/FS))
trim_table = pd.DataFrame(rows)
trim_table["removed_s"] = trim_table.before_s - trim_table.after_s
print(trim_table.round(3).head(10)); print(trim_table[["before_s", "after_s"]].mean().round(3))

In [ ]:
# Kiểm tra từ bắt đầu bằng phụ âm năng lượng thấp ("khong" bắt đầu bằng "kh")
f = train_files["khong"][0]; y = load_audio(f)
fig, axes = plt.subplots(3, 1, figsize=(10, 8))
settings = [("Không ZCR refine, margin 0", dict(zcr_refine=False, margin_frames=0)),
            ("ZCR refine, margin 0",       dict(zcr_refine=True,  margin_frames=0)),
            (f"ZCR refine + margin {EP_MARGIN_FRAMES} frame (dùng trong Lab)", dict())]
for ax, (name, kw) in zip(axes, settings):
    s, e, _ = endpoint_detect(y, **kw)
    ax.plot(np.arange(len(y))/FS, y, lw=.6)
    ax.axvline(s/FS, color="g"); ax.axvline(e/FS, color="r")
    ax.set_title(f"{name}: start={s/FS:.3f}s end={e/FS:.3f}s"); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig(FIG/"C_endpoint_khong_margin.png", dpi=150); plt.show()

**Nhận xét C:** (điền) Thời lượng trung bình trước/sau trim; có cắt mất âm đầu "kh" hay âm cuối không (hãy **nghe lại** các file trong thư mục `trimmed/`); margin/ngưỡng cuối cùng chọn là bao nhiêu và vì sao.

## D. MFCC

Pre-emphasis ($\alpha=0.97$) → Hamming → FFT 512 → Mel filterbank 24 bộ lọc → log → DCT, giữ 13 hệ số, kèm CMN theo từng utterance. Mọi tham số dùng chung cho train và test.

In [ ]:
def mfcc_feature(y, delta=False, cmn=True):
    y = lfilter([1.0, -PREEMPH], [1.0], y)                     # pre-emphasis
    M = librosa.feature.mfcc(y=y, sr=FS, n_mfcc=N_MFCC, n_mels=N_MELS,
                             n_fft=NFFT, win_length=WIN, hop_length=HOP,
                             window="hamming", center=False)
    if cmn:
        M = M - np.mean(M, axis=1, keepdims=True)              # cepstral mean normalization
    F = M.T                                                    # (T frames, 13)
    if delta:
        d = librosa.feature.delta(M, width=5, mode="nearest").T
        F = np.hstack([F, d])                                  # (T, 26)
    return F

# Mel filterbank
fb = librosa.filters.mel(sr=FS, n_fft=NFFT, n_mels=N_MELS)
plt.figure(figsize=(10, 3.5))
for m in range(N_MELS):
    plt.plot(np.linspace(0, FS/2, fb.shape[1]), fb[m])
plt.title("Mel filterbank 24 bộ lọc, Fs = 16 kHz"); plt.xlabel("Tần số (Hz)"); plt.ylabel("Trọng số")
plt.tight_layout(); plt.savefig(FIG/"D_mel_filterbank.png", dpi=150); plt.show()

In [ ]:
# Heatmap MFCC của 2 từ khác nhau
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, lab in zip(axes, [LABELS[0], LABELS[1]]):
    y = trim_endpoint(load_audio(train_files[lab][0]))
    F = mfcc_feature(y)
    im = ax.imshow(F.T, origin="lower", aspect="auto", cmap="magma")
    ax.set_title(f"MFCC '{VN[lab]}'  shape={F.shape}")
    ax.set_xlabel("Frame"); ax.set_ylabel("Hệ số MFCC"); plt.colorbar(im, ax=ax)
plt.tight_layout(); plt.savefig(FIG/"D_mfcc_heatmap.png", dpi=150); plt.show()

# Số frame khác nhau, số chiều cố định
for lab in LABELS:
    shapes = [mfcc_feature(trim_endpoint(load_audio(f))).shape for f in list_files(lab)]
    print(lab, shapes)

**Vì sao số frame khác nhau nhưng số chiều MFCC/frame cố định?** Số frame $T = 1+\lfloor (N-L)/R\rfloor$ phụ thuộc độ dài tín hiệu $N$ (tốc độ nói), còn mỗi frame luôn được biến đổi thành đúng 13 hệ số (chọn bởi `n_mfcc`) bất kể độ dài. Vì thế mẫu từ là ma trận $T\times 13$ với $T$ thay đổi — đây chính là lý do cần DTW để so sánh hai chuỗi có độ dài khác nhau.

## E. DTW tự cài đặt

Local distance Euclid, $D[i,j]=C[i,j]+\min\{D[i-1,j],D[i,j-1],D[i-1,j-1]\}$, backtrack lấy optimal path, chuẩn hoá $D[N-1,M-1]/|P|$.

In [ ]:
def local_distance_matrix(X, Y):
    # C[i,j] = ||x_i - y_j||_2 (tự tính bằng NumPy)
    diff = X[:, None, :] - Y[None, :, :]
    return np.sqrt(np.sum(diff ** 2, axis=2))

def dtw(X, Y, return_all=False):
    N, M = len(X), len(Y)
    C = local_distance_matrix(X, Y)
    D = np.full((N + 1, M + 1), np.inf)
    D[0, 0] = 0.0
    back = np.zeros((N + 1, M + 1, 2), dtype=int)
    for i in range(1, N + 1):
        for j in range(1, M + 1):
            cands = ((D[i-1, j], i-1, j),       # bước dọc
                     (D[i, j-1], i, j-1),       # bước ngang
                     (D[i-1, j-1], i-1, j-1))   # bước chéo
            best, pi, pj = min(cands, key=lambda z: z[0])
            D[i, j] = C[i-1, j-1] + best
            back[i, j] = (pi, pj)
    path, i, j = [], N, M
    while i > 0 or j > 0:
        path.append((i-1, j-1))
        i, j = back[i, j]
    path.reverse()
    dist = D[N, M] / max(len(path), 1)
    if return_all:
        return dist, path, C, D[1:, 1:]
    return dist, path

# Kiểm tra bắt buộc
Xt = np.random.RandomState(0).randn(30, 13)
d0, _ = dtw(Xt, Xt)
print("DTW(X, X) =", d0, "(phải ≈ 0)")
assert d0 < 1e-9

# Ví dụ số trong đề: x=[1,2], y=[4,6] → d=5
print("Euclid ví dụ:", local_distance_matrix(np.array([[1., 2.]]), np.array([[4., 6.]]))[0, 0])

In [ ]:
@lru_cache(maxsize=None)
def _feat_cached(path_str, trim, delta):
    y = load_audio(path_str)
    if trim:
        y = trim_endpoint(y)
    return mfcc_feature(y, delta=delta)

def get_feature(path, trim=True, delta=False):
    return _feat_cached(str(path), trim, delta)

def plot_dtw(X, Y, C, path, title, fname):
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(C, origin="lower", aspect="auto", cmap="Blues")
    pi, pj = zip(*path)
    ax.plot(pj, pi, color="orangered", lw=2)
    ax.set_xlabel("Frame j của mẫu Y"); ax.set_ylabel("Frame i của mẫu X")
    ax.set_title(title); plt.colorbar(im, label="Khoảng cách Euclid")
    plt.tight_layout(); plt.savefig(FIG/fname, dpi=150); plt.show()

lab = LABELS[1]
X  = get_feature(train_files[lab][0]); Ys = get_feature(train_files[lab][1])
Yd = get_feature(train_files[LABELS[2]][0])

d_same, p_same, C_same, _ = dtw(X, Ys, return_all=True)
d_diff, p_diff, C_diff, _ = dtw(X, Yd, return_all=True)
plot_dtw(X, Ys, C_same, p_same, f"Cùng từ '{VN[lab]}': DTW_norm = {d_same:.2f}", "E_dtw_same.png")
plot_dtw(X, Yd, C_diff, p_diff, f"Khác từ '{VN[lab]}' vs '{VN[LABELS[2]]}': DTW_norm = {d_diff:.2f}", "E_dtw_diff.png")
print(f"DTW_norm cùng từ  = {d_same:.3f}  (|P|={len(p_same)})")
print(f"DTW_norm khác từ = {d_diff:.3f}  (|P|={len(p_diff)})")

**Nhận xét E:** Với cùng một từ, đường đi bám tương đối gần đường chéo (có đoạn ngang/dọc do tốc độ nói khác nhau) và DTW_norm nhỏ hơn rõ rệt so với hai từ khác nhau, khi đó đường đi phải "uốn" nhiều hơn để né các ô có chi phí cao và tổng chi phí tăng. *(Đối chiếu với số liệu in ở trên của bạn.)*

## F. Bộ nhận dạng nearest-template

Với mỗi test: trích MFCC → DTW tới toàn bộ template → $D_w(X)=\min_r \mathrm{DTW}_{norm}$ → $\hat w=\arg\min_w D_w$.

In [ ]:
def build_templates(files_by_label, n_templates=N_TRAIN, trim=True, delta=False):
    return {lab: [get_feature(f, trim, delta) for f in files_by_label[lab][:n_templates]]
            for lab in LABELS}

def recognize_feature(X, templates):
    scores = {lab: min(dtw(X, R)[0] for R in refs) for lab, refs in templates.items()}
    pred = min(scores, key=scores.get)
    return pred, dict(sorted(scores.items(), key=lambda kv: kv[1]))

def recognize_file(path, templates, trim=True, delta=False):
    # API "file WAV chưa biết → nhãn + khoảng cách DTW" theo yêu cầu đầu ra
    return recognize_feature(get_feature(path, trim, delta), templates)

templates = build_templates(train_files)

rows = []
for lab in LABELS:
    for f in test_files[lab]:
        pred, sc = recognize_file(f, templates)
        top = list(sc.items())
        rows.append(dict(file=f.name, true=lab, pred=pred,
                         top1=top[0][0], d1=top[0][1], top2=top[1][0], d2=top[1][1],
                         top3=top[2][0], d3=top[2][1]))
results = pd.DataFrame(rows)
results.round(3)

In [ ]:
results[["file", "true", "pred", "d1", "d2"]].rename(columns={"d1": "top1_dtw", "d2": "top2_dtw"}) \
    .to_csv("results.csv", index=False)
results.head(10).round(3)

In [ ]:
# (Tùy chọn) reject 'unknown': chọn θ = mean + 2*std của khoảng cách top-1 các mẫu đã nhận đúng
ok = results[results.true == results.pred]
theta = ok.d1.mean() + 2 * ok.d1.std()
rej = (results.d1 > theta)
print(f"θ = {theta:.3f} (mean+2std của top-1 các mẫu đúng); số mẫu bị reject = {rej.sum()}/{len(results)}")

## G. Đánh giá và thí nghiệm

In [ ]:
def evaluate(trim=True, delta=False, n_templates=N_TRAIN, train=train_files, test=test_files):
    tpl = build_templates(train, n_templates, trim, delta)
    yt, yp = [], []
    for lab in LABELS:
        for f in test[lab]:
            pred, _ = recognize_file(f, tpl, trim, delta)
            yt.append(lab); yp.append(pred)
    return yt, yp

yt, yp = evaluate()
acc = accuracy_score(yt, yp)
cm = confusion_matrix(yt, yp, labels=LABELS)
print(f"Accuracy = {acc*100:.1f}%  ({sum(a==b for a,b in zip(yt,yp))}/{len(yt)})")
print(pd.DataFrame(cm, index=LABELS, columns=LABELS))

def plot_cm(cm, title, fname):
    fig, ax = plt.subplots(figsize=(5, 4.5))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(LABELS))); ax.set_xticklabels(LABELS, rotation=30)
    ax.set_yticks(range(len(LABELS))); ax.set_yticklabels(LABELS)
    for i in range(len(LABELS)):
        for j in range(len(LABELS)):
            ax.text(j, i, cm[i, j], ha="center", va="center",
                    color="white" if cm[i, j] > cm.max()/2 else "black")
    ax.set_xlabel("Dự đoán"); ax.set_ylabel("Nhãn thật"); ax.set_title(title)
    plt.colorbar(im); plt.tight_layout(); plt.savefig(FIG/fname, dpi=150); plt.show()

plot_cm(cm, f"Confusion matrix (acc={acc*100:.1f}%)", "G_confusion_matrix.png")

# cặp từ dễ nhầm nhất
off = cm.copy(); np.fill_diagonal(off, 0)
if off.max() > 0:
    i, j = np.unravel_index(off.argmax(), off.shape)
    print(f"Cặp dễ nhầm nhất: '{LABELS[i]}' bị nhận thành '{LABELS[j]}' ({off[i, j]} lần)")
else:
    print("Không có nhầm lẫn nào trên tập test.")

### Thí nghiệm E1–E3 (mỗi thí nghiệm chỉ đổi đúng một yếu tố)

In [ ]:
exp = []
def run(name, **kw):
    a, b = evaluate(**kw)
    exp.append(dict(experiment=name, accuracy=100*accuracy_score(a, b)))

run("E1a: Không trim",  trim=False)
run("E1b: Có trim",     trim=True)
run("E2a: 13 MFCC",     delta=False)
run("E2b: 13 MFCC + Δ", delta=True)
run("E3a: 1 template/từ", n_templates=1)
run("E3b: 3 template/từ", n_templates=3)
exp_table = pd.DataFrame(exp).round(1)
exp_table.to_csv("experiments.csv", index=False)
exp_table

In [ ]:
# E4 (nếu có dữ liệu người nói thứ hai): thư mục dataset_speaker2/ cùng cấu trúc
SPK2 = Path("dataset_speaker2")
if SPK2.exists():
    test2 = {lab: list_files(lab, SPK2) for lab in LABELS}          # toàn bộ file người 2 làm test
    a, b = evaluate(test=test2)
    print(f"Cross-speaker accuracy: {100*accuracy_score(a, b):.1f}%")
    print(f"Same-speaker  accuracy: {100*acc:.1f}%")
else:
    print("Không có dataset_speaker2/ → bỏ qua E4 (chỉ làm nếu ghi âm đủ hai người).")

**Nhận xét thí nghiệm (điền theo bảng `exp_table` của bạn):**
- *E1:* Không trim thì DTW phải căn chỉnh cả đoạn silence, chi phí tăng và dễ nhầm hơn.
- *E2:* Δ (động học) thêm thông tin biến thiên theo thời gian; so sánh với 13 MFCC tĩnh.
- *E3:* Nhiều template giúp bao phủ biến thiên tốc độ/cách phát âm tốt hơn.

## Trả lời câu hỏi báo cáo

1. **Vì sao không dùng toàn bộ waveform làm template?** Hai utterance khác độ dài và lệch pha, nên so sánh mẫu-theo-mẫu không có ý nghĩa; waveform còn chứa pha và biến thiên nhỏ không liên quan nội dung. Dùng đặc trưng theo frame (MFCC) + DTW cho phép so sánh phổ ngắn hạn sau khi căn chỉnh thời gian.
2. **Vai trò energy và ZCR:** Energy tách vùng có tiếng nói khỏi nền (voiced năng lượng lớn). ZCR bổ sung cho các âm vô thanh/fricative có năng lượng thấp nhưng ZCR cao, giúp tinh chỉnh biên để không cắt mất phụ âm đầu/cuối.
3. **Vì sao Mel filterbank rộng dần theo Hz?** Thang Mel xấp xỉ cảm nhận thính giác: phân giải cao ở tần số thấp, thô ở tần số cao, nên bộ lọc đều trên thang Mel sẽ có độ rộng tăng dần theo Hz.
4. **Log và DCT:** Log nén dynamic range (năng lượng phổ chênh nhau nhiều bậc độ lớn) và biến tích chập nguồn–bộ lọc thành phép cộng. DCT biến M log-energy filterbank thành các hệ số cepstral (MFCC), tương quan thấp hơn; các hệ số đầu mô tả đường bao phổ.
5. **Bước ngang/dọc/chéo:** Chéo = hai frame ứng với nhau, tốc độ tương đương; ngang = một frame của Y được ghép với nhiều frame X liên tiếp (X bị kéo dãn so với Y hay Y bị nén); dọc = ngược lại.
6. **Vì sao chuẩn hoá theo |P|?** Tổng chi phí cộng dồn theo số cặp frame trên đường đi, nên utterance dài tự nhiên có chi phí lớn hơn; chia cho |P| đưa về chi phí trung bình mỗi bước để so sánh công bằng.
7. **Nguyên nhân MFCC khác nhau giữa hai lần nói cùng từ:** (i) tốc độ/nhịp nói và co giãn thời gian; (ii) khác biệt phát âm, ngữ điệu, cách nhấn; (iii) khoảng cách micro, mức âm lượng, nhiễu nền, kênh thu; (iv) sai số endpoint detection; (v) khác người nói (giọng, vocal tract).
8. **Cặp từ dễ nhầm:** xem kết quả "Cặp dễ nhầm nhất" ở phần G, rồi vẽ waveform/MFCC/DTW path của cặp đó để đề xuất nguyên nhân (ví dụ vần/nguyên âm tương tự, từ ngắn, endpoint cắt sai).
9. **Hạn chế với người nói mới:** DTW chỉ so khớp với template đã lưu nên không khái quát hoá cho người nói/kênh thu mới và khó mở rộng từ vựng lớn. Chương 3 (mô hình thống kê HMM, mô hình âm học/ngôn ngữ, huấn luyện trên nhiều người nói) giải quyết tốt hơn.

## Kết luận
(Tóm tắt accuracy, bảng thí nghiệm E1–E3 (và E4 nếu có), cặp từ hay nhầm, đề xuất cải thiện.)